# LivePortrait idle motion through ComfyUI-AdvancedLivePortrait - birthday test set

**Author**: Konrad Jelen (kj) <br>
**Date**: 2026-10-24 <br>
**Model**: LivePortrait through ComfyUI-AdvancedLivePortrait, node pack `PowerHouseMan/ComfyUI-AdvancedLivePortrait@3bba732`, weights `Kijai/LivePortrait_safetensors@59f30f3` <br>
**Photo set**: `henryk-urodziny-2026`, three working copies <br>

Renders 5-second live-photo clips of three birthday photos of Henry with LivePortrait driven by keyframed expressions instead of a driving clip: a slight smile that builds slowly and then varies a little in strength, one natural blink and three very small head movements with short holds between them. Every photo is rendered in three variants (`gentle`, `soft`, `still`) and every clip is timed and measured, so the Star Colonel can view the clips side by side and judge them; the notebook measures and does not rank.

The AdvancedLivePortrait node works on LivePortrait's implicit 3D keypoints of the photo: canonical keypoints x_c,s, expression offsets δ_s, head rotation angles θ_s (pitch, yaw, roll), scale s_s and translation t_s. Each keyframe k is an ExpressionEditor node that turns its slider values into an expression change Δδ_k (the smile and blink sliders move fixed keypoints by fixed amounts per slider unit) and a rotation change Δθ_k in degrees. For an output frame i between the keyframe frames f_k-1 < i ≤ f_k the node blends linearly

$$\Delta_i = \Delta_{k-1} + \frac{i - f_{k-1}}{f_k - f_{k-1}} \left( \Delta_k - \Delta_{k-1} \right)$$

and renders the keypoints

$$x_{d,i} = s_s \left( x_{c,s} + \delta_s + \Delta\delta_i \right) R\left( \theta_s + \Delta\theta_i \right) + t_s$$

- **Scale and translation stay the photo's** - s_s and t_s are the same in every frame, so the head never moves forward or backward and never changes size
- **Stitching and paste-back** - stitching aligns the animated 512 x 512 face crop with the shoulders; paste-back blends the crop into the full-resolution photo through a fixed mask, so pixels outside the mask keep the photo's values
- **One face per photo** - the node animates the face whose YOLO box (confidence ≥ 0.7, width ≥ 30 px) has its centre nearest the photo's horizontal centre

The keyframes sample eased curves. A curve c runs through knots (f_k, v_k) and eases from each knot to the next with the smootherstep S(x) = 6x⁵ − 15x⁴ + 10x³ (x clamped to [0, 1], zero slope at both ends), so every movement starts and stops with zero speed; before the first knot and after the last it holds the knot's value:

$$c(i) = v_k + \left( v_{k+1} - v_k \right) S\!\left( \frac{i - f_k}{f_{k+1} - f_k} \right), \qquad f_k \le i < f_{k+1}$$

With the peak values M (smile), Y (yaw), P (pitch) and R (roll) of a variant, frame i gets

$$\mathrm{smile}_i = M \, c_{\mathrm{smile}}(i), \quad \mathrm{yaw}_i = Y \, c_{\mathrm{turn}}(i), \quad \mathrm{roll}_i = R \, c_{\mathrm{turn}}(i), \quad \mathrm{pitch}_i = P \, c_{\mathrm{pitch}}(i)$$

- **Smile** - knots (frame, fraction of M) (10, 0), (60, 1), (75, 1), (100, 0.7), (130, 1): builds over 0.33-2.00 s, holds, eases back to 70 % of its peak by 3.33 s and returns to the peak by 4.33 s
- **Head** - three movements with 0.5 s holds between them: frames 5-40 (1.17 s) turn to Y and P/2, frames 55-95 (1.33 s) turn across to −Y/2 and 0.8 P, frames 110-145 (1.17 s) turn back to yaw 0 and P. Roll follows the turn, so the head tilts slightly with each turn
- **Chin** - pitch only grows, and a positive pitch lowers the chin, so the chin never lifts
- **Blink** - one per clip: the lids close over frames 60-64, stay closed on frames 64-65 and open over frames 65-71, 0.37 s from open to open
- **Keyframes** - every 5 frames (0.17 s), every knot, the last frame and the blink frames; between keyframes this close together the linear blend follows the eased curves

## Approach
1. **Pin** the RTX PRO 5000 Blackwell (nvidia-smi index 2) by UUID for the ComfyUI server - every time and memory figure comes from one known card
2. **Start** the ComfyUI server on 127.0.0.1 and show the device it reports - ComfyUI runs in its own environment with a CUDA 13.0 PyTorch build
3. **Load** the three working copies and report size and colour profile - each clip keeps its photo's resolution
4. **Warm up** with LivePortrait's public example image `s0.jpg` - the first render loads the LivePortrait weights and the face detector, a cost kept out of the measured clips and out of the family photos
5. **Render** 3 photos × 3 variants, restoring finished clips from their checkpoints - 9 clips
6. **Clean up and stop** - every copy of a photo in ComfyUI's input, output and temp folders goes to the trash after its render, and the server stops when the renders are done
7. **Measure** wall time, peak VRAM and the region of the photo that changed, show which face moved in the three-face photo, and show every clip under its photo - the Star Colonel judges the clips himself

## Outputs
- `data/processed/henryk-urodziny-2026/advanced-liveportrait/<stem>--<variant>.mp4` - 9 clips, 150 frames at 30 fps (5.00 s), at the photo's own resolution
- `<stem>--<variant>.md` sidecar per clip (source working copy, model and commits, expression values, GPU, seconds, peak VRAM) and `checkpoints/<stem>--<variant>.json` with the same record for restore on re-run
- `IMG_8564--faces.jpg` - the face boxes the node finds in the three-face photo drawn on a copy of it, the animated face in green
- In-notebook: configuration, working-copy table, keyframe tables, face table of the three-face photo, results table, changed-region table, and every clip under its photo, linked by path so no photo or video bytes are stored in the notebook

## GPU Selection

Pins the RTX PRO 5000 72GB Blackwell (nvidia-smi index 2, compute capability 12.0) by UUID before the project package imports torch. The ComfyUI server that does the rendering is started with the same pin; this kernel itself makes no CUDA call. The Configuration render below shows the device the server reports.

In [ ]:
import os

# Deterministic ordering so a UUID/index always names the same physical card
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# RTX PRO 5000 72GB Blackwell, pinned by UUID from `nvidia-smi -L`; the ComfyUI server gets the same pin
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-1223ff00-ce71-874f-592e-a4429e9f47ba"

## Imports

All imports in one cell. The project package comes after the GPU pin because `henryk_media.animation.common` imports torch. The ComfyUI client talks to the server over HTTP and imports nothing from ComfyUI.

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import tempfile  # Scratch folder for the discarded warm-up clip
from dataclasses import asdict  # Variant peak values as table rows
from pathlib import Path  # File path handling

# Notebook display
from IPython.display import Image, Markdown, Video, display  # Photos and clips shown by path, never embedded

# Rich console output
from rich import box  # Table border style
from rich import print as rprint  # Formatted console printing
from rich.console import Group  # Several tables in one output block
from rich.progress import (  # Progress bar components
    BarColumn,
    MofNCompleteColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)
from rich.table import Table  # Tabular output

# Project package
from henryk_media.animation import advanced_liveportrait as alp  # ComfyUI server, keyframes, render
from henryk_media.animation import common  # Clip paths, sidecars, checkpoints, changed region
from henryk_media.animation.liveportrait import save_face_overlay  # Box drawn on a copy of a photo
from henryk_media.config import INTERIM_DATA_DIR, MODELS_DIR  # Project paths

## Reproducibility

Seeds Python, NumPy and PyTorch in this kernel by convention. The render runs in the ComfyUI server and draws no random numbers: the keyframes are fixed curves, the face detector and the LivePortrait networks are deterministic forward passes.

In [ ]:
common.set_seed(42)

## Configuration

Inputs, the three variants and the ComfyUI server. The cell ends by starting the server, so its render can show the device the server reports.

- **Variants** - peak values in the ExpressionEditor's units: smile slider, degrees of yaw, pitch and roll, blink slider. `gentle` stays below the research recipe (smile 0.2, yaw 3, pitch −1); `soft` is somewhat larger; `still` has no blink and no expression, only the three head movements at the smallest size
- **Pitch sign** - on LivePortrait's public example images `s0.jpg`, `s5.jpg` and `s42.jpg`, pitch +4 moved the facial features 4.4-7.2 px down and pitch −4 moved them 4.3-7.1 px up (median optical flow in the middle third of the face). Positive pitch lowers the chin, so every variant uses a positive pitch; the research recipe's −1 lifted the chin
- **Blink −16** - on LivePortrait's public example images viewed at blink −4 to −20, the lids met at about −14 on the adult photos `s0.jpg` and `s42.jpg` and on the child photo `s11.jpg`, and at −16 to −17 on the painted child `s5.jpg`, whose eyes are the largest. The node moves the lid keypoints in proportion to the slider with no stop at the closed position, so values past the closing point press the closed lids further down. −16 closes or almost closes all four; both blinking variants use it, because a lighter value leaves large eyes visibly open
- **Changed-region threshold** - a pixel counts as changed when, in some frame, it differs from frame 0 by more than 8 on the 0-255 scale (mean over the colour channels)

In [ ]:
# Inputs
PHOTO_SET = "henryk-urodziny-2026"  # folder name under data/interim and data/processed
WORKING_COPIES = [  # PNG working copies with EXIF orientation applied; originals stay in data/external
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8840.png",
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8564.png",
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8732.png",
]
THREE_FACE_PHOTO = WORKING_COPIES[1]  # IMG_8564: three children side by side; the node animates one face
WARMUP_IMAGE = MODELS_DIR / "photo-animation/LivePortrait/assets/examples/source/s0.jpg"  # public example, not a family photo

# Output
OUTPUT_DIR = common.output_dir(PHOTO_SET, "advanced-liveportrait")  # clips, sidecars, checkpoints/, overlay

# Variants: peak values in the ExpressionEditor's units (smile slider, degrees, blink slider)
VARIANTS = {
    "gentle": alp.Motion(smile=0.15, yaw=2.0, pitch=0.5, roll=0.3, blink=-16.0),  # below the research recipe
    "soft": alp.Motion(smile=0.25, yaw=3.0, pitch=1.0, roll=0.5, blink=-16.0),  # somewhat larger, still subtle
    "still": alp.Motion(smile=0.0, yaw=1.0, pitch=0.3, roll=0.0, blink=0.0),  # tiny head arc only
}

# Measurement
CHANGE_THRESHOLD = 8.0  # 0-255; changed = differs from frame 0 by more than this in some frame

# ComfyUI server on the pinned GPU, listening on 127.0.0.1 only
server = alp.ComfyUI()
device, system = server.stats["devices"][0], server.stats["system"]

variant_lines = "\n".join(
    f"  [cadet_blue]{name}[/cadet_blue]: "
    + ", ".join(f"{key} [grey70]{value:g}[/grey70]" for key, value in asdict(motion).items())
    for name, motion in VARIANTS.items()
)
knots = alp.settings(VARIANTS["soft"])  # curve knots as the sidecars record them
photo_lines = "\n".join(f"  [cadet_blue]{photo.name}[/cadet_blue]" for photo in WORKING_COPIES)
rprint(f'''[bold medium_purple]Configuration[/bold medium_purple]
[dim]{"─" * 40}[/dim]
[bold slate_blue1]Inputs[/bold slate_blue1]
  Working copies: [dim]{WORKING_COPIES[0].parent}[/dim]
{photo_lines}
  Three-face photo: [cadet_blue]{THREE_FACE_PHOTO.name}[/cadet_blue]
  Warm-up image: [dim]{WARMUP_IMAGE}[/dim]

[bold slate_blue1]Variants[/bold slate_blue1] [dim](peak values)[/dim]
{variant_lines}

[bold slate_blue1]Timeline[/bold slate_blue1]
  Clip: [dark_sea_green]{alp.FRAMES}[/dark_sea_green] frames at [light_sea_green]{alp.FPS} fps[/light_sea_green] ([light_sea_green]{alp.FRAMES / alp.FPS:.2f} s[/light_sea_green])
  Keyframe step: [dark_sea_green]{alp.KEYFRAME_STEP}[/dark_sea_green] frames
  Smile knots: [grey70]{knots['smile_knots']}[/grey70] [dim](frame:fraction of the peak)[/dim]
  Head knots: [grey70]{knots['yaw_knots']}[/grey70] yaw, [grey70]{knots['pitch_knots']}[/grey70] pitch, [grey70]{knots['roll_knots']}[/grey70] roll [dim](frame:fraction of each axis's peak)[/dim]
  Blink: frames [dark_sea_green]{alp.BLINK_START}-{alp.BLINK_START + alp.BLINK_SHAPE[-1][0]}[/dark_sea_green] [dim](closed on {", ".join(str(alp.BLINK_START + offset) for offset, closed in alp.BLINK_SHAPE if closed == 1)})[/dim]
  Crop factor: [grey70]{alp.CROP_FACTOR}[/grey70]

[bold slate_blue1]ComfyUI[/bold slate_blue1]
  Version: [grey70]{system["comfyui_version"]}[/grey70], PyTorch [grey70]{system["pytorch_version"]}[/grey70]
  Server: [cadet_blue]{alp.SERVER}[/cadet_blue] [dim]({" ".join(alp.LAUNCH_ARGS[2:])})[/dim]
  Log: [dim]{alp.SERVER_LOG}[/dim]

[bold slate_blue1]Output[/bold slate_blue1]
  Clips: [dim]{OUTPUT_DIR}[/dim]
  Clip count: [dark_sea_green]{len(WORKING_COPIES) * len(VARIANTS)}[/dark_sea_green]

[bold]Device[/bold] [dim](reported by the ComfyUI server)[/dim]
  Using: [green]{device["type"]}:{device["index"]}[/green]
  GPU: [cyan]{device["name"]}[/cyan]
  Pinned UUID: [cyan]{alp.GPU_UUID}[/cyan]
  Memory: [yellow]{device["vram_total"] / 1e9:.1f} GB[/yellow]
''')

## Data Loading

Reads each working copy for its size and embedded colour profile. The node keeps the photo's resolution, so these sizes are the clip sizes.

In [ ]:
photo_table = Table(title="Working copies", box=box.SIMPLE_HEAVY)
for column in ("Photo", "Width x height", "Colour profile"):
    photo_table.add_column(column, style="grey70" if column == "Photo" else None)
for photo in WORKING_COPIES:
    height, width = common.load_working_copy(photo).shape[:2]
    photo_table.add_row(
        photo.name, f"[light_sea_green]{width} x {height} px[/light_sea_green]", common.colour_profile(photo) or "none"
    )
rprint(photo_table)

## Model

The server loads the LivePortrait networks and the YOLO face detector on the first render, not at start. The first cell renders LivePortrait's public example image `s0.jpg` once with the `soft` variant and discards the clip, so that one-time cost is reported on its own and no family photo carries it. The second cell lists the keyframes each variant submits.

In [ ]:
# Discarded warm-up render: its clip, sidecar and checkpoint go to a scratch folder removed on exit
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(dir=OUTPUT_DIR) as scratch:
    warmup = server.render(WARMUP_IMAGE, VARIANTS["soft"], "soft", Path(scratch) / "warmup.mp4")
rprint(
    f"[bold medium_purple]Warm-up render[/bold medium_purple] [dim](discarded)[/dim]\n"
    f"  Image: [cadet_blue]{WARMUP_IMAGE.name}[/cadet_blue] [dim]({warmup.width} x {warmup.height} px, {warmup.frames} frames)[/dim]\n"
    f"  Revision: [cadet_blue]{server.revision}[/cadet_blue]\n"
    f"  Time: [light_sea_green]{warmup.seconds:.1f} s[/light_sea_green]\n"
    f"  Peak VRAM: [light_sea_green]{warmup.peak_vram_gb:.2f} GB[/light_sea_green] [dim](above the memory in use before the server started)[/dim]"
)

In [ ]:
summary = Table(title="Variants", box=box.SIMPLE_HEAVY)
for column in ("Variant", "Smile", "Yaw", "Pitch", "Roll", "Blink", "Keyframes"):
    summary.add_column(column, justify="left" if column == "Variant" else "right")
for name, motion in VARIANTS.items():
    summary.add_row(
        f"[grey70]{name}[/grey70]",
        f"{motion.smile:g}",
        f"{motion.yaw:g}°",
        f"{motion.pitch:g}°",
        f"{motion.roll:g}°",
        f"{motion.blink:g}" if motion.blink else "[dim]none[/dim]",
        f"[dark_sea_green]{len(alp.keyframes(motion))}[/dark_sea_green]",
    )

schedule = Table(
    title="Keyframes of the soft variant",
    caption="gentle follows the same curves scaled to its peak values; still has no smile, no blink and no blink keyframes",
    box=box.SIMPLE_HEAVY,
)
for column in ("Frame", "Time", "Smile", "Yaw", "Pitch", "Roll", "Blink"):
    schedule.add_column(column, justify="right")
for frame, values in alp.keyframes(VARIANTS["soft"]):
    schedule.add_row(
        f"[grey70]{frame}[/grey70]",
        f"[light_sea_green]{frame / alp.FPS:.2f} s[/light_sea_green]",
        *(f"{values[key]:.3f}" for key in ("smile", "rotate_yaw", "rotate_pitch", "rotate_roll")),
        f"[dark_goldenrod]{values['blink']:.1f}[/dark_goldenrod]" if values["blink"] else "0",
    )
rprint(Group(summary, schedule))

## Rendering

Renders each working copy in each variant: 3 photos × 3 variants = 9 clips. For each clip the client copies the photo into ComfyUI's input folder, submits the graph, moves the mp4 to the output folder and moves every file the render added to ComfyUI's input, output and temp folders to the trash. Time runs from submission to the finished history entry; peak VRAM is sampled with nvidia-smi during the render. A clip whose video and checkpoint exist with the same settings (peak values, curve knots and keyframes) is restored from the checkpoint instead of rendered again. The second cell runs the node's face detection once more on the three-face photo, without rendering a clip, and reads from the server log how many faces YOLO found and which one the node animates.

In [ ]:
jobs = [(photo, variant) for photo in WORKING_COPIES for variant in VARIANTS]
results, restored = [], set()

with Progress(
    SpinnerColumn(),
    TextColumn("[steel_blue]{task.description}"),
    BarColumn(bar_width=40),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
    refresh_per_second=10,
) as progress:
    task = progress.add_task("Rendering clips", total=len(jobs))
    for photo, variant in jobs:
        clip = common.clip_path(OUTPUT_DIR, photo, variant)
        progress.update(task, description=f"Rendering {clip.name}")
        result = alp.restore(clip, VARIANTS[variant])
        if result is None:
            result = server.render(photo, VARIANTS[variant], variant, clip)
        else:
            restored.add(result.clip)
        results.append(result)
        progress.advance(task)
    # force completion (prevents N-1 issue)
    progress.update(task, completed=len(jobs), description="Rendered clips")
    progress.refresh()

In [ ]:
# The node's own face detection on the three-face photo; its log lines name the face it animates
face_count, face_boxes = server.detect_faces(THREE_FACE_PHOTO)
ordered_boxes = [face_boxes[-1], *face_boxes[:-1]]  # animated face first, numbered like the overlay below
centre_x = common.load_working_copy(THREE_FACE_PHOTO).shape[1] / 2

face_table = Table(
    title=f"Faces the node prints for {THREE_FACE_PHOTO.name} (YOLO face count {face_count})",
    caption="the node keeps the box whose centre lies nearest the photo's horizontal centre",
    box=box.SIMPLE_HEAVY,
)
for column in ("Box", "x0 - x1", "y0 - y1", "Centre distance", "Node"):
    face_table.add_column(column, justify="left" if column in ("Box", "Node") else "right")
for number, (x0, y0, x1, y1) in enumerate(ordered_boxes, start=1):
    face_table.add_row(
        f"[grey70]{number}[/grey70]",
        f"{x0} - {x1}",
        f"{y0} - {y1}",
        f"[light_sea_green]{abs((x0 + x1) / 2 - centre_x):.0f} px[/light_sea_green]",
        "[dark_sea_green]animated[/dark_sea_green]" if number == 1 else "[dim]still[/dim]",
    )
rprint(face_table)


## Save/Export

Each render already wrote its clip, sidecar and checkpoint. This cell lists the output folder, lists every file left in ComfyUI's input, output and temp folders to show that no copy of a photo or clip remains there, and stops the ComfyUI server.

In [ ]:
written = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
comfyui_files = sorted(alp.work_files())
server.stop()
file_lines = "\n".join(
    f"  [cadet_blue]{path.relative_to(OUTPUT_DIR)}[/cadet_blue] [dim]{path.stat().st_size / 1e6:.2f} MB[/dim]" for path in written
)
comfyui_lines = "\n".join(f"  [cadet_blue]{path.relative_to(alp.COMFYUI_DIR)}[/cadet_blue]" for path in comfyui_files)
stopped = not alp.server_answers()
rprint(
    f"[bold medium_purple]Output folder[/bold medium_purple] [dim]{OUTPUT_DIR}[/dim]\n{file_lines}\n\n"
    f"[bold slate_blue1]Files in ComfyUI's {', '.join(alp.WORK_FOLDERS)} folders[/bold slate_blue1] [dim](ComfyUI's own files only)[/dim]\n"
    f"{comfyui_lines or '  [dim]none[/dim]'}\n\n"
    f"[bold slate_blue1]ComfyUI server[/bold slate_blue1]\n"
    f"  Stopped: [{'dark_sea_green' if stopped else 'indian_red'}]{'✓ yes' if stopped else '✗ no'}[/] "
    f"[dim](exit code {server.process.returncode})[/dim]"
)

## Results

Measurements per clip, the region of each photo that changed, and the clips themselves under their source photos.

- **Time** - wall seconds from submitting the graph to the finished history entry: face detection, one ExpressionEditor preview per keyframe, 150 frames of warping, decoding and paste-back at the photo's resolution, and the H.264 encode
- **Peak VRAM** - the highest nvidia-smi `memory.used` of the GPU sampled during the render, minus the memory in use before the server started; it is the memory of the ComfyUI process (loaded weights, activations and CUDA context) as long as other processes on the card hold their memory constant
- **Changed region** - the bounding box of the pixels p whose largest difference from frame 0 exceeds the threshold, after a 5 x 5 opening removes isolated pixels:

$$\max_{1 \le i \le 149} \; \operatorname{mean}_{c} \left| I_i(p, c) - I_0(p, c) \right| > 8$$

In [ ]:
result_table = Table(title="AdvancedLivePortrait clips", box=box.SIMPLE_HEAVY)
for column in ("Clip", "Variant", "Seconds", "Peak VRAM", "Size", "Frames", "FPS", "Duration", "Measured"):
    result_table.add_column(column, justify="left" if column in ("Clip", "Variant", "Measured") else "right")
for result in results:
    result_table.add_row(
        f"[grey70]{Path(result.clip).name}[/grey70]",
        result.variant,
        f"[light_sea_green]{result.seconds:.1f} s[/light_sea_green]",
        f"[light_sea_green]{result.peak_vram_gb:.2f} GB[/light_sea_green]",
        f"{result.width} x {result.height}",
        f"[dark_sea_green]{result.frames}[/dark_sea_green]",
        f"[dark_sea_green]{result.fps:g}[/dark_sea_green]",
        f"[light_sea_green]{result.frames / result.fps:.2f} s[/light_sea_green]",
        "[dim]checkpoint[/dim]" if result.clip in restored else "this run",
    )
rprint(result_table)

In [ ]:
regions = {}
with Progress(
    SpinnerColumn(),
    TextColumn("[steel_blue]{task.description}"),
    BarColumn(bar_width=40),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
    refresh_per_second=10,
) as progress:
    task = progress.add_task("Measuring changed regions", total=len(jobs))
    for photo, variant in jobs:
        regions[photo, variant] = common.changed_region(common.clip_path(OUTPUT_DIR, photo, variant), CHANGE_THRESHOLD)
        progress.advance(task)
    progress.update(task, completed=len(jobs), description="Measured changed regions")
    progress.refresh()

In [ ]:
region_table = Table(title=f"Region that changed against frame 0 (threshold {CHANGE_THRESHOLD:g})", box=box.SIMPLE_HEAVY)
for column in ("Clip", "x0 - x1", "y0 - y1", "Width x height", "Share of photo"):
    region_table.add_column(column, justify="left" if column == "Clip" else "right")
for result, ((photo, variant), region) in zip(results, regions.items()):
    if region is None:
        region_table.add_row(f"[grey70]{Path(result.clip).name}[/grey70]", "[dim]none[/dim]", "", "", "")
        continue
    x0, y0, x1, y1 = region
    region_table.add_row(
        f"[grey70]{Path(result.clip).name}[/grey70]",
        f"{x0} - {x1}",
        f"{y0} - {y1}",
        f"[light_sea_green]{x1 - x0} x {y1 - y0} px[/light_sea_green]",
        f"[dark_sea_green]{(x1 - x0) * (y1 - y0) / (result.width * result.height):.1%}[/dark_sea_green]",
    )
rprint(region_table)

In [ ]:
# Face boxes the node found, drawn on a copy of the three-face photo; shown by path, no photo bytes stored here
overlay = common.clip_path(OUTPUT_DIR, THREE_FACE_PHOTO, "faces").with_suffix(".jpg")
save_face_overlay(THREE_FACE_PHOTO, ordered_boxes, overlay)
display(Markdown(f"**{THREE_FACE_PHOTO.name}** - box `1 animated` (green) is the face the node animates; red boxes stay still"))
display(Image(url=os.path.relpath(overlay), width=480))


In [ ]:
# Mean change of each face box against frame 0 in the clips of the three-face photo, numbered like the overlay
face_change_table = Table(
    title=f"Mean change per face box in the {THREE_FACE_PHOTO.stem} clips (0-255 scale, against frame 0)",
    box=box.SIMPLE_HEAVY,
)
face_change_table.add_column("Clip")
for number in range(1, len(ordered_boxes) + 1):
    face_change_table.add_column(f"Box {number}" + (" (animated)" if number == 1 else ""), justify="right")
for variant in VARIANTS:
    clip = common.clip_path(OUTPUT_DIR, THREE_FACE_PHOTO, variant)
    changes = common.region_motion(clip, ordered_boxes)
    face_change_table.add_row(
        f"[grey70]{clip.name}[/grey70]", *(f"[dark_sea_green]{change:.2f}[/dark_sea_green]" for change in changes)
    )
rprint(face_change_table)


**Which face moved in the three-face photo** - the middle face. YOLO found 3 faces in IMG_8564, and the node animates box 1 (x 410-625, y 276-546), whose centre lies 36 px from the photo's horizontal centre; the right face (box 2, 396 px from the centre) and the left face (box 3, 269 px) stay still.

- **Measured change** - inside box 1 the mean change against frame 0 is 2.23 (`still`), 3.80 (`gentle`) and 5.39 (`soft`) on the 0-255 scale; inside boxes 2 and 3 it is 0.04-0.06 in all three clips. Both lie outside the node's crop square (box 3 overlaps its left edge by 13 px), where every frame keeps the photo's pixels and only the H.264 encode differs between frames
- **Changed region** - the node replaces a square crop of side 1.7 × the face box's longer side, centred on the box: 1.7 × 270 px = 459 px, x 287-746, y 180-639 for box 1. The changed regions of the three IMG_8564 clips (x 293-743, y 183-593 at the widest) lie inside that square


In [ ]:
# Photo and clips linked by path relative to the notebook: no image or video bytes are stored here
for photo in WORKING_COPIES:
    display(Markdown(f"### {photo.stem}"))
    display(Image(url=os.path.relpath(photo), width=400))
    for variant, motion in VARIANTS.items():
        clip = common.clip_path(OUTPUT_DIR, photo, variant)
        values = ", ".join(f"{key} {value:g}" for key, value in asdict(motion).items())
        display(Markdown(f"`{variant}` - {values}"))
        display(Video(os.path.relpath(clip), embed=False, width=400, html_attributes="controls loop muted"))

**Measurements** - from the tables above; all 9 clips were rendered in this execution, none restored from a checkpoint.

- **Frames** - every clip has 150 frames at 30 fps (5.00 s) at its photo's resolution
- **Time** - 14.7-16.3 s for the 1.2-1.4 megapixel photos IMG_8564 and IMG_8840, 24.8-25.7 s for the 3.0 megapixel IMG_8732. The variant changes the time of one photo by at most 1.6 s (IMG_8564: 14.7 s `gentle`, 16.3 s `soft` and `still`). The node renders the face at a fixed 512 x 512 px for every photo, but pastes each of the 150 frames and each keyframe preview back into the full-resolution photo, and VideoHelperSuite encodes the full-resolution frames, so the photo's pixel count sets the time
- **Peak VRAM** - 2.20-2.54 GB above the memory in use before the server started (warm-up 2.17 GB). The value grew in render order from 2.20 GB on the first clip to 2.54 GB on the ninth and never fell; the server ran all 9 renders without a restart, and this notebook did not trace the growth to a cause
- **Changed region** - 32.3-38.7 % of the photo for IMG_8840, 11.0-15.1 % for IMG_8564 and IMG_8732. The node changes pixels only inside its crop square of 1.7 × the face box's longer side; the face in IMG_8840 fills a larger share of its photo, so its square does too. The `still` clip has the smallest region of each photo: it has no smile, no blink and the smallest head turn


## Conclusions

The three birthday working copies are rendered in three variants each with the new timeline: the smile builds and then eases back slightly and returns, the head makes three short movements with holds between them, and the eyes blink once. How natural the motion looks is not measured here; the Star Colonel judges the clips above.

- **Rendered** - 9 of 9 clips, each 150 frames at 30 fps, each with a sidecar and a checkpoint; no cell raised an error
- **Smile** - the `soft` keyframe table shows the smile reaching 0.250 at 2.00 s, easing back to 0.175 (70 %) at 3.33 s and returning to 0.250 at 4.33 s; `gentle` follows the same curve with a 0.15 peak, `still` has no smile
- **Head** - in the `soft` table yaw reaches 3.000° at 1.33 s, −1.500° at 3.17 s and 0 at 4.83 s, with 0.5 s holds at 1.33-1.83 s and 3.17-3.67 s. Pitch only grows, from 0 to 1.000°, and a positive pitch lowers the chin, so the chin never lifts. The node keeps the photo's scale and translation in every frame, so the head does not move forward or backward
- **Blink** - one blink in `gentle` and `soft`: blink −16 on frames 64-65, 0.37 s from open to open; `still` has no blink keyframe
- **Three-face photo** - only the middle face moves: mean change 2.23-5.39 inside its box against 0.04-0.06 inside the two other face boxes
- **Cost** - 14.7-25.7 s and 2.20-2.54 GB per clip on the RTX PRO 5000; the photo's pixel count sets the time, which keeps a re-render of all 9 clips under 3 minutes of render time
- **Privacy** - after the run ComfyUI's input, output and temp folders hold only ComfyUI's own two files, and the server stopped (exit code −15: ended by the SIGTERM signal that the client's `stop` sends)
